In [12]:
import pandas as pd
import numpy as np
from dython.nominal import associations
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt

# 다중공선성 확인을 위햔 라이브러리
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 'Malgun Gothic' 폰트를 기본으로 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

In [2]:
df = pd.read_parquet('C:/Users/이민정/Desktop/부트캠프/파이널프로젝트/전처리/AB추출.parquet')

In [3]:
na_counts = df.isna().sum()
na_cols = na_counts[na_counts >0]
print(na_cols)

가입통신회사코드            79
직장시도명               17
_2순위신용체크구분         234
최종유효년월_신용_이용         7
RV신청일자             889
연체일자_B0M          1116
OS구분코드             474
혜택수혜율_R3M            9
혜택수혜율_B0M           13
_1순위업종              22
_2순위업종              39
_3순위업종              62
_1순위쇼핑업종            45
_2순위쇼핑업종            71
_3순위쇼핑업종            99
_1순위교통업종           101
_2순위교통업종           216
_3순위교통업종           436
_1순위여유업종           326
_2순위여유업종           636
_3순위여유업종           863
_1순위납부업종           189
_2순위납부업종           550
_3순위납부업종           892
최종카드론_금융상환방식코드     642
최종카드론_신청경로코드       644
최종카드론_대출일자         768
dtype: int64


In [4]:
df = df.dropna(axis=1)
df = df.drop(['ID'],axis=1)

In [5]:
df.head()

,기준년월,남녀구분코드,연령,Segment,회원여부_이용가능,회원여부_이용가능_CA,회원여부_이용가능_카드론,소지여부_신용,소지카드수_유효_신용,소지카드수_이용가능_신용,...,승인거절건수_한도초과_B0M,승인거절건수_BL_B0M,승인거절건수_입력오류_B0M,승인거절건수_기타_B0M,승인거절건수_R3M,승인거절건수_한도초과_R3M,승인거절건수_BL_R3M,승인거절건수_입력오류_R3M,승인거절건수_기타_R3M,이용금액대
0,201807,1,40대,A,1,1,0,1,1,1,...,0,0,0,0,3,3,0,0,0,01.100만원+
1,201807,2,50대,A,1,1,1,1,2,2,...,0,0,0,0,0,0,0,0,0,01.100만원+
2,201807,2,40대,A,1,1,1,1,3,3,...,0,0,0,0,0,0,0,0,0,01.100만원+
3,201807,1,40대,A,1,1,1,1,3,3,...,0,0,0,0,5,0,4,0,1,01.100만원+
4,201807,2,30대,A,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,01.100만원+


In [6]:
# LabelEncoder 하기

obj_cols = df.select_dtypes(include='object').columns

le_dict = {}

for col in obj_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1116 entries, 0 to 1115
Columns: 830 entries, 기준년월 to 이용금액대
dtypes: float64(54), int32(29), int64(747)
memory usage: 6.9 MB


In [7]:
correlations = df.corr()["Segment"].sort_values(ascending=False)

In [8]:
significant_corr = correlations[correlations.abs() >= 0.35]
significant_corr_sorted = significant_corr.reindex(significant_corr.abs().sort_values(ascending=False).index)

print(significant_corr_sorted)

Segment          1.000000
카드이용한도금액_B1M    -0.653554
카드이용한도금액_B2M    -0.645587
카드이용한도금액        -0.635069
CA한도금액          -0.515938
이용건수_선결제_R6M     0.402885
선결제건수_R6M        0.382041
상환개월수_결제일_R6M   -0.373127
이용건수_선결제_R3M     0.372249
이용금액_선결제_R6M     0.371204
선입금원금_B2M        0.370902
이용횟수_선결제_R6M     0.370574
선입금원금_B0M        0.368353
상환개월수_결제일_R3M   -0.363593
선결제건수_R3M        0.360349
이용금액_선결제_R3M     0.358689
선입금원금_B5M        0.356789
이용개월수_결제일_R6M   -0.351421
Name: Segment, dtype: float64


In [23]:
X = df[['카드이용한도금액_B1M','카드이용한도금액_B2M','카드이용한도금액','CA한도금액',
         '이용건수_선결제_R6M','선결제건수_R6M','상환개월수_결제일_R6M','이용건수_선결제_R3M',
        '이용금액_선결제_R6M','선입금원금_B2M','이용횟수_선결제_R6M','선입금원금_B0M','상환개월수_결제일_R3M',
        '선결제건수_R3M','이용금액_선결제_R3M','선입금원금_B5M','이용개월수_결제일_R6M']]
X= add_constant(X)

In [25]:
vif = pd.DataFrame()
vif["변수"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

               변수        VIF
0           const  66.449432
1    카드이용한도금액_B1M  13.709569
2    카드이용한도금액_B2M  12.506364
3        카드이용한도금액  14.803126
4          CA한도금액   2.140725
5    이용건수_선결제_R6M   6.994332
6       선결제건수_R6M   9.527276
7   상환개월수_결제일_R6M   9.513420
8    이용건수_선결제_R3M   3.085032
9    이용금액_선결제_R6M  12.328713
10      선입금원금_B2M   6.541799
11   이용횟수_선결제_R6M  13.634454
12      선입금원금_B0M   6.527878
13  상환개월수_결제일_R3M   8.590486
14      선결제건수_R3M   8.001166
15   이용금액_선결제_R3M  15.583864
16      선입금원금_B5M   5.109794
17  이용개월수_결제일_R6M   2.608616


In [65]:
X1 = df[['카드이용한도금액','CA한도금액',
         '이용건수_선결제_R3M',
        '선입금원금_B2M','상환개월수_결제일_R3M',
        '선입금원금_B5M','이용개월수_결제일_R6M']]
X1 = add_constant(X1)

In [67]:
vif = pd.DataFrame()
vif["변수"] = X1.columns
vif["VIF"] = [variance_inflation_factor(X1.values, i) for i in range(X1.shape[1])]
print(vif)

              변수        VIF
0          const  55.731940
1       카드이용한도금액   2.071584
2         CA한도금액   2.027919
3   이용건수_선결제_R3M   1.731900
4      선입금원금_B2M   3.857328
5  상환개월수_결제일_R3M   3.007551
6      선입금원금_B5M   3.302890
7  이용개월수_결제일_R6M   2.429176
